# Semana 3: Tarea

**Valoración DCF de tu empresa** (la misma que elegiste en la tarea de la semana 2)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JonathanRosasV/topicos-finanzas-upao/blob/main/03_dcf_fcff_fcfe/clase03_tarea.ipynb)

**Entrega: hasta el lunes de la semana siguiente, vía `git commit` + `push` en tu fork** (`03_dcf_fcff_fcfe/clase03_tarea.ipynb`)

Total: 20 puntos (18 de contenido y 2 de forma: el notebook corre de inicio a fin y la entrega llega a tiempo).

## Parte 1: FCFF histórico por dos rutas (4 pts)

Con los estados financieros de tu empresa (`yfinance`), calcula el FCFF de los últimos 3 a 4 años por la ruta del EBIT y por la ruta del CFO, en una tabla comparativa. Comenta en una o dos oraciones por qué las rutas no coinciden exactamente.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import pandas as pd
import yfinance as yf
from utils.finanzas import (fcff_desde_ebit, fcff_desde_cfo, fcfe_desde_fcff,
                            valor_crecimiento_constante, capm, wacc)

TICKER = "IFS"   # tu empresa de la semana 2

# TU CODIGO AQUI: estados financieros, insumos y FCFF por ambas rutas
def fila(df, *nombres):
    """Devuelve la primera fila disponible entre los nombres candidatos."""
    for n in nombres:
        if n in df.index:
            return df.loc[n]
    raise KeyError(f"Ninguna de estas filas existe: {nombres}")

tk = yf.Ticker(TICKER)

est = tk.income_stmt
bal = tk.balance_sheet
cf = tk.cashflow

ebit = fila(est, "Pretax Income") + fila(est, "Interest Expense").abs()
interes = fila(est, "Interest Expense").abs()
impuesto = fila(est, "Tax Provision")
util_at = fila(est, "Pretax Income")
dep = fila(cf, "Depreciation And Amortization",
           "Depreciation Amortization Depletion",
           "Depreciation")
capex = fila(cf, "Capital Expenditure").abs()
d_wc = -fila(cf, "Change In Working Capital")
cfo = fila(cf, "Operating Cash Flow")

t_efectiva = float((impuesto / util_at).iloc[0])
print(f"Tasa efectiva de impuestos (ultimo anio): {t_efectiva:.1%}")

resumen = pd.DataFrame({
    "EBIT": ebit,
    "Dep": dep,
    "Capex": capex,
    "WCInv": d_wc,
    "Interes": interes,
    "CFO": cfo
}).dropna() / 1e6

resumen = resumen.sort_index(ascending=False)
resumen.round(0)

anios = resumen.index

fcff_ebit = pd.Series(
    [fcff_desde_ebit(
        resumen.loc[a, "EBIT"],
        t_efectiva,
        resumen.loc[a, "Dep"],
        resumen.loc[a, "Capex"],
        resumen.loc[a, "WCInv"]
    ) for a in anios],
    index=anios,
    name="FCFF (ruta EBIT)"
)

fcff_cfo = pd.Series(
    [fcff_desde_cfo(
        resumen.loc[a, "CFO"],
        resumen.loc[a, "Interes"],
        t_efectiva,
        resumen.loc[a, "Capex"]
    ) for a in anios],
    index=anios,
    name="FCFF (ruta CFO)"
)

comparacion = pd.concat([fcff_ebit, fcff_cfo], axis=1)
comparacion["diferencia %"] = (fcff_ebit / fcff_cfo - 1) * 100

comparacion.round(1)

Tasa efectiva de impuestos (ultimo anio): 21.8%


,FCFF (ruta EBIT),FCFF (ruta CFO),diferencia %
2025-12-31,2429.4,2922.1,-16.9
2024-12-31,2977.6,4376.7,-32.0
2023-12-31,2752.5,4371.6,-37.0
2022-12-31,-776.0,-176.5,339.8


In [2]:
resumen.round(1)

,EBIT,Dep,Capex,WCInv,Interes,CFO
2025-12-31,4499.8,450.3,523.4,1017.6,2064.4,1830.6
2024-12-31,3968.3,413.1,350.1,189.7,2322.3,2910.0
2023-12-31,3806.4,379.0,428.0,176.2,2460.0,2875.2
2022-12-31,3697.5,336.2,362.3,3642.3,1531.3,-1012.1


In [3]:
print(est.index.tolist())

['Tax Effect Of Unusual Items', 'Tax Rate For Calcs', 'Total Unusual Items', 'Total Unusual Items Excluding Goodwill', 'Net Income From Continuing Operation Net Minority Interest', 'Reconciled Depreciation', 'Net Interest Income', 'Interest Expense', 'Interest Income', 'Normalized Income', 'Net Income From Continuing And Discontinued Operation', 'Rent Expense Supplemental', 'Diluted Average Shares', 'Basic Average Shares', 'Diluted EPS', 'Basic EPS', 'Diluted NI Availto Com Stockholders', 'Net Income Common Stockholders', 'Net Income', 'Minority Interests', 'Net Income Including Noncontrolling Interests', 'Net Income Extraordinary', 'Net Income Continuous Operations', 'Tax Provision', 'Pretax Income', 'Special Income Charges', 'Other Special Charges', 'Write Off', 'Impairment Of Capital Assets', 'Restructuring And Mergern Acquisition', 'Gain On Sale Of Security', 'Depreciation Amortization Depletion Income Statement', 'Depreciation And Amortization In Income Statement', 'Selling Genera

Las dos rutas no coinciden porque el CFO ya incorpora otros ajustes que no son caja y cambios operativos que no aparecen de forma separada en la ruta del EBIT. En IFS la diferencia también puede ser mayor porque al ser financiera, el EBIT tuvo que aproximarse a partir de la utilidad antes de impuestos más los intereses.

En 2022 la diferencia porcentual es muy alta porque ambas rutas dan valores negativos y el FCFF por CFO está relativamente cerca de cero, además de que el aumento del capital de trabajo fue elevado.

## Parte 2: proyección con supuestos declarados (4 pts)

Proyecta el FCFF a 5 años. Declara tus supuestos de crecimiento en una celda de texto y justifícalos en una o dos oraciones (historia de la empresa, sector, o crecimiento de la economía). El crecimiento perpetuo no puede superar el crecimiento nominal de largo plazo de la economía donde opera.

In [4]:
# TU CODIGO AQUI: proyeccion a 5 anios y crecimiento perpetuo declarado
fcff_0 = float(fcff_ebit.iloc[0])
print(f"Anio base: {fcff_ebit.index[0].year} | FCFF base = {fcff_0:,.0f} MM")

g_proyeccion = [0.06, 0.055, 0.05, 0.045, 0.04]
g_perpetuo = 0.03

proy = []
f = fcff_0

for g in g_proyeccion:
    f = f * (1 + g)
    proy.append(f)

proy = pd.Series(proy, index=range(1, 6), name="FCFF proyectado")

print(proy.round(1))

Anio base: 2025 | FCFF base = 2,429 MM
1    2575.1
2    2716.8
3    2852.6
4    2981.0
5    3100.2
Name: FCFF proyectado, dtype: float64


Se tomó como base el FCFF de 2025 y se proyectó con tasas de crecimiento de 6%, 5.5%, 5%, 4.5% y 4% para los siguientes cinco años. Se usan tasas cada vez menores porque se espera que el crecimiento de IFS se vaya estabilizando y para el largo plazo se considera un crecimiento perpetuo de 3%.

## Parte 3: valoración completa (4 pts)

Con el WACC de tu tarea de la semana 2: valor presente de los flujos, valor terminal, EV, puente a equity (resta la deuda neta) y valor por acción. Reporta el peso del valor terminal en el EV y compara tu valor por acción contra el precio de mercado.

In [5]:
# TU CODIGO AQUI: VT, EV, equity, valor por accion y comparacion con el precio
# WACC obtenido en la tarea de la semana 2
WACC = 0.1090

# Valor terminal y valor presente
VT5 = valor_crecimiento_constante(
    proy.iloc[-1] * (1 + g_perpetuo),
    WACC,
    g_perpetuo
)

vp_flujos = sum(
    cf_i / (1 + WACC) ** i
    for i, cf_i in proy.items()
)

vp_VT = VT5 / (1 + WACC) ** 5
EV = vp_flujos + vp_VT

print(f"VP de los 5 flujos      = {vp_flujos:,.1f} MM")
print(f"VP del valor terminal   = {vp_VT:,.1f} MM")
print(f"EV                      = {EV:,.1f} MM")
print(f"Peso del valor terminal = {vp_VT/EV:.1%}")

caja = float(
    fila(
        bal,
        "Cash And Cash Equivalents",
        "Cash Cash Equivalents And Short Term Investments"
    ).iloc[0]
) / 1e6

D = float(fila(bal, "Total Debt").iloc[0]) / 1e6
deuda_neta = D - caja

acciones = tk.fast_info["shares"] / 1e6

eq = EV - deuda_neta
precio_modelo_pen = eq / acciones

# Convertimos PEN por acción a USD por acción
fx = yf.download(
    "PEN=X",
    period="5d",
    auto_adjust=True,
    progress=False
)["Close"].dropna()

tipo_cambio = float(fx.iloc[-1].iloc[0])

precio_modelo_usd = precio_modelo_pen / tipo_cambio
precio_mercado = tk.fast_info["lastPrice"]

print(f"EV = {EV:,.1f} MM PEN")
print(f"Deuda neta = {deuda_neta:,.1f} MM PEN")
print(f"Equity = {eq:,.1f} MM PEN")
print(f"Valor por accion modelo = US$ {precio_modelo_usd:,.2f}")
print(f"Precio de mercado       = US$ {precio_mercado:,.2f}")
print(f"Diferencia              = {precio_modelo_usd/precio_mercado - 1:+.1%}")

VP de los 5 flujos      = 10,441.3 MM
VP del valor terminal   = 24,095.8 MM
EV                      = 34,537.1 MM
Peso del valor terminal = 69.8%
EV = 34,537.1 MM PEN
Deuda neta = -1,392.8 MM PEN
Equity = 35,929.9 MM PEN
Valor por accion modelo = US$ 95.90
Precio de mercado       = US$ 54.98
Diferencia              = +74.4%


El valor presente de los flujos proyectados es 10,441.3 MM PEN y el valor presente del valor terminal es 24,095.8 MM PEN, dando un EV de 34,537.1 MM PEN. El valor terminal representa 69.8% del EV, por lo que gran parte de la valoración depende de los supuestos de largo plazo. Al restar la deuda neta se obtiene un equity de 35,929.9 MM PEN y un valor estimado de US$95.90 por acción, frente a un precio de mercado de US$54.98, una diferencia de aproximadamente 74.4%

Esta diferencia no debe interpretarse como una ganancia segura, porque el resultado depende de los supuestos de crecimiento, WACC y del valor terminal.

## Parte 4: veredicto profesional (2 pts)

En un párrafo: ¿tu modelo sugiere que la acción está subvaluada, sobrevaluada o bien valorada? Nombra los dos supuestos de tu modelo que más te preocupan y en qué dirección sesgarían el resultado si estuvieran mal.

Mi modelo sugiere que la acción de IFS está subvaluada, porque el valor estimado es de US$95.90 por acción y el precio de mercado es de US$54.98. Los dos supuestos que más me preocupan son el WACC de 10.9% y el crecimiento perpetuo de 3%, porque ambos influyen bastante en la valoración. Si el WACC usado fuera demasiado bajo, el modelo estaría sobreestimando el valor de la acción; y si el crecimiento perpetuo fuera demasiado alto, también estaría inflando el valor estimado.

## Parte 5: ítems tipo CFA (4 pts, 1 c/u)

**1.** Al calcular el FCFF desde la utilidad neta, el ajuste correcto por intereses es sumar:

A. el gasto por intereses completo. B. el gasto por intereses después de impuestos. C. nada, los intereses no se tocan.

**2.** Una empresa reporta FCFF = 120, interés = 30, t = 25% y endeudamiento neto = -10 (amortizó deuda). Su FCFE es:

A. 87.5. B. 97.5. C. 107.5.

**3.** El valor terminal en un DCF de dos etapas típicamente representa:

A. una fracción menor del valor total. B. entre 60 y 80 por ciento del valor total. C. exactamente el 50 por ciento del valor.

**4.** Un analista descuenta el FCFE de una empresa al WACC. Su valoración del equity estará, en general:

A. sobreestimada, porque el WACC es menor que el costo del equity. B. subestimada. C. correcta, ambas tasas son intercambiables.

**Respuestas:**

1.B. El gasto por intereses después de impuestos, porque al pasar de utilidad neta a FCFF se devuelve. Interés × (1 - t).
2.B. 97.5. FCFE = 120 - 30(1 - 0.25) - 10 = 97.5
3.B. El valor terminal suele representar entre 60% y 80% del valor total en un DCF de dos etapas.
4.A. En general estará sobreestimada, porque el FCFE debe descontarse al Ke, no al WACC; normalmente el WACC es menor que el costo del equity.

---

**Recuerda:** `Kernel  Restart & Run All` antes de entregar, y luego:

```bash
git add 03_dcf_fcff_fcfe/clase03_tarea.ipynb
git commit -m "Tarea semana 3"
git push
```